In [ ]:
!pip install -q timm albumentations lmdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 11.0 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import lmdb
import timm
import torch
import pickle
import random
import numpy as np
import pandas as pd

from tqdm import tqdm
from collections import defaultdict

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.utils.class_weight import compute_class_weight

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torch import nn
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class CFG:

    IMG_SIZE = 224

    BATCH_SIZE = 32

    EPOCHS = 5

    LR = 1e-4

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    NUM_WORKERS = 2

    BASE_DIR = "/content/drive/MyDrive/deepfake_project"

    TRAIN_CSV = os.path.join(BASE_DIR, "data/metadata/train.csv")

    TEST_CSV = os.path.join(BASE_DIR, "data/metadata/test.csv")

    # FAST SSD PATHS
    TRAIN_LMDB = "/content/train.lmdb"

    TEST_LMDB = "/content/test.lmdb"

    # DRIVE STORAGE
    DRIVE_TRAIN_LMDB = os.path.join(BASE_DIR, "train.lmdb")

    DRIVE_TEST_LMDB = os.path.join(BASE_DIR, "test.lmdb")

In [ ]:
train_df = pd.read_csv(CFG.TRAIN_CSV)

test_df = pd.read_csv(CFG.TEST_CSV)

print(len(train_df))
print(len(test_df))

print(train_df.head())

25696
6362
                                          frame_path  video_id  label
0  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
1  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
2  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
3  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0
4  /content/drive/MyDrive/deepfake_project/data/c...  real/000      0


In [ ]:
# verify video level split
train_videos = set(train_df["video_id"].unique())

test_videos = set(test_df["video_id"].unique())

intersection = train_videos.intersection(test_videos)

print("Overlap videos:", len(intersection))

assert len(intersection) == 0, "VIDEO LEAKAGE DETECTED!"

Overlap videos: 0


In [ ]:
if os.path.exists(CFG.DRIVE_TRAIN_LMDB):

    print("Copying train LMDB from Drive...")

    !cp "{CFG.DRIVE_TRAIN_LMDB}" /content/

if os.path.exists(CFG.DRIVE_TEST_LMDB):

    print("Copying test LMDB from Drive...")

    !cp "{CFG.DRIVE_TEST_LMDB}" /content/

Copying train LMDB from Drive...
Copying test LMDB from Drive...


In [ ]:
# Augmentations

train_transform = A.Compose([

    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.ImageCompression(
        quality_range=(60, 100),
        p=0.5
    ),

    A.GaussianBlur(
        blur_limit=(3, 5),
        p=0.3
    ),

    A.Normalize(),

    ToTensorV2()
])

valid_transform = A.Compose([

    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.Normalize(),

    ToTensorV2()
])

In [ ]:
class DeepfakeLMDBDataset(Dataset):

    def __init__(self, lmdb_path, transform=None):

        self.lmdb_path = lmdb_path

        self.env = None

        self.txn = None

        self.transform = transform

        env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        self.keys = []

        self.labels = []

        with env.begin() as txn:

            cursor = txn.cursor()

            for key, value in cursor:

                self.keys.append(key)

                sample = pickle.loads(value)

                self.labels.append(int(sample["label"]))

        env.close()

    def _init_db(self):

        if self.env is None:

            self.env = lmdb.open(
                self.lmdb_path,
                readonly=True,
                lock=False,
                readahead=False,
                meminit=False,
                subdir=False
            )

            self.txn = self.env.begin()

    def __len__(self):

        return len(self.keys)

    def __getitem__(self, idx):

        self._init_db()

        key = self.keys[idx]

        value = self.txn.get(key)

        sample = pickle.loads(value)

        img_bytes = sample["image"]

        label = sample["label"]

        video_id = sample["video_id"]

        # convert bytes to numpy array
        img_array = np.frombuffer(img_bytes, np.uint8)

        # decode image
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        # convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:

            img = self.transform(image=img)["image"]

        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.float32),
            "video_id": video_id
        }

In [ ]:
# create datasets
train_dataset = DeepfakeLMDBDataset(
    CFG.TRAIN_LMDB,
    transform=train_transform
)

test_dataset = DeepfakeLMDBDataset(
    CFG.TEST_LMDB,
    transform=valid_transform
)

print(len(train_dataset))
print(len(test_dataset))

25696
6362


In [ ]:
# weighted sampler
class_counts = np.bincount(train_dataset.labels)

weights = 1. / class_counts

sample_weights = [
    weights[label]
    for label in train_dataset.labels
]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
# DataLoaders (optimized for collab gpu)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    sampler=sampler,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

In [ ]:
# EfficientNetV2 Model
model = timm.create_model(
    "tf_efficientnetv2_b0",
    pretrained=True,
    num_classes=1
)

model = model.to(CFG.DEVICE)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/28.8M [00:00<?, ?B/s]

Model loaded.


In [ ]:
# Loss, Optimizer and AMP
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR
)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_720/3612227089.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
# Training function
def train_one_epoch(model, loader):

    model.train()

    running_loss = 0

    all_preds = []

    all_labels = []

    pbar = tqdm(loader)

    for batch in pbar:

        images = batch["image"].to(CFG.DEVICE)

        labels = batch["label"].to(CFG.DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            logits = model(images).squeeze(1)

            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        probs = torch.sigmoid(logits)

        preds = (probs > 0.5).float()

        all_preds.extend(preds.detach().cpu().numpy())

        all_labels.extend(labels.detach().cpu().numpy())

        pbar.set_description(
            f"Loss: {loss.item():.4f}"
        )

    acc = accuracy_score(all_labels, all_preds)

    return running_loss / len(loader), acc

In [ ]:
# Evaluation Function (stores IDs for later video-level aggregation)
def evaluate(model, loader):

    model.eval()

    all_probs = []

    all_preds = []

    all_labels = []

    all_video_ids = []

    with torch.no_grad():

        for batch in tqdm(loader):

            images = batch["image"].to(CFG.DEVICE)

            labels = batch["label"].to(CFG.DEVICE)

            video_ids = batch["video_id"]

            with torch.cuda.amp.autocast():

                logits = model(images).squeeze(1)

            probs = torch.sigmoid(logits)

            preds = (probs > 0.5).float()

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_video_ids.extend(video_ids)

    acc = accuracy_score(all_labels, all_preds)

    precision = precision_score(all_labels, all_preds)

    recall = recall_score(all_labels, all_preds)

    f1 = f1_score(all_labels, all_preds)

    auc = roc_auc_score(all_labels, all_probs)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "probs": all_probs,
        "labels": all_labels,
        "video_ids": all_video_ids
    }

In [ ]:
best_auc = 0

for epoch in range(CFG.EPOCHS):

    print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    metrics = evaluate(
        model,
        test_loader
    )

    print(f"Train Loss: {train_loss:.4f}")

    print(f"Train Acc: {train_acc:.4f}")

    print(f"Test Acc: {metrics['accuracy']:.4f}")

    print(f"Precision: {metrics['precision']:.4f}")

    print(f"Recall: {metrics['recall']:.4f}")

    print(f"F1: {metrics['f1']:.4f}")

    print(f"AUC: {metrics['auc']:.4f}")

    if metrics["auc"] > best_auc:

        best_auc = metrics["auc"]

        torch.save(
            model.state_dict(),
            "/content/best_model.pth"
        )

        print("Best model saved.")


Epoch 1/5


  0%|          | 0/803 [00:00<?, ?it/s]/tmp/ipykernel_720/1138155629.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  0%|          | 0/199 [00:00<?, ?it/s]/tmp/ipykernel_720/3232262701.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 199/199 [00:13<00:00, 14.51it/s]


Train Loss: 0.1908
Train Acc: 0.9350
Test Acc: 0.9348
Precision: 0.7879
Recall: 0.7871
F1: 0.7875
AUC: 0.9681
Best model saved.

Epoch 2/5


  0%|          | 0/803 [00:00<?, ?it/s]/tmp/ipykernel_720/1138155629.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  0%|          | 0/199 [00:00<?, ?it/s]/tmp/ipykernel_720/3232262701.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 199/199 [00:12<00:00, 16.48it/s]


Train Loss: 0.0277
Train Acc: 0.9899
Test Acc: 0.9453
Precision: 0.8467
Recall: 0.7861
F1: 0.8153
AUC: 0.9756
Best model saved.

Epoch 3/5


...

Train Loss: 0.0156
Train Acc: 0.9947
Test Acc: 0.9533
Precision: 0.9250
Recall: 0.7574
F1: 0.8329
AUC: 0.9819
Best model saved.

Epoch 4/5


  0%|          | 0/803 [00:00<?, ?it/s]/tmp/ipykernel_720/1138155629.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  0%|          | 0/199 [00:00<?, ?it/s]/tmp/ipykernel_720/3232262701.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 199/199 [00:12<00:00, 16.39it/s]


Train Loss: 0.0136
Train Acc: 0.9951
Test Acc: 0.9513
Precision: 0.8669
Recall: 0.8066
F1: 0.8356
AUC: 0.9825
Best model saved.

Epoch 5/5


  0%|          | 0/803 [00:00<?, ?it/s]/tmp/ipykernel_720/1138155629.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  0%|          | 0/199 [00:00<?, ?it/s]/tmp/ipykernel_720/3232262701.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 199/199 [00:10<00:00, 19.35it/s]


Train Loss: 0.0089
Train Acc: 0.9968
Test Acc: 0.9541
Precision: 0.8844
Recall: 0.8066
F1: 0.8437
AUC: 0.9832
Best model saved.


In [ ]:
# save best model to drive
!cp /content/best_model.pth \
"/content/drive/MyDrive/deepfake_project/"

In [ ]:
# Loads the best checkpoint (based on AUC), switches model to inference mode and prepares for evaluation
model.load_state_dict(torch.load("/content/best_model.pth"))
model.eval()
print("Best model loaded for video-level evaluation.")

Best model loaded for video-level evaluation.


In [ ]:
metrics = evaluate(model, test_loader)

probs = np.array(metrics["probs"]) # frame-level fake/real probabilities
labels = np.array(metrics["labels"]) # tells which frame belongs to which video
video_ids = np.array(metrics["video_ids"]) # frame labels (not used for final video metric, but useful)

  0%|          | 0/199 [00:00<?, ?it/s]/tmp/ipykernel_720/3232262701.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 199/199 [00:14<00:00, 13.60it/s]


In [ ]:
# video level aggregation (average probability per video)
video_dict = defaultdict(list)

for vid, prob in zip(video_ids, probs):
    video_dict[vid].append(prob)

In [ ]:
video_preds = []
video_labels = []

for vid, frame_probs in video_dict.items():

    avg_prob = np.mean(frame_probs)

    video_pred = 1 if avg_prob > 0.5 else 0

    video_preds.append(video_pred)

    # take label of first frame (all frames same video label)
    video_labels.append(labels[list(video_ids).index(vid)])

In [ ]:
video_acc = accuracy_score(video_labels, video_preds)
video_prec = precision_score(video_labels, video_preds)
video_rec = recall_score(video_labels, video_preds)
video_f1 = f1_score(video_labels, video_preds)

print("\n===== VIDEO-LEVEL RESULTS =====")
print(f"Accuracy:  {video_acc:.4f}")
print(f"Precision: {video_prec:.4f}")
print(f"Recall:    {video_rec:.4f}")
print(f"F1 Score:  {video_f1:.4f}")


===== VIDEO-LEVEL RESULTS =====
Accuracy:  0.9787
Precision: 0.9697
Recall:    0.8889
F1 Score:  0.9275


Majority Voting for Comparison with Average Probability Method:

In [ ]:
# Majority Voting Version

video_preds_mv = []

for vid, frame_probs in video_dict.items():

    frame_preds = [1 if p > 0.5 else 0 for p in frame_probs]

    final = int(np.round(np.mean(frame_preds)))  # majority vote

    video_preds_mv.append(final)

In [ ]:
video_acc_mv = accuracy_score(video_labels, video_preds_mv)
video_prec_mv = precision_score(video_labels, video_preds_mv)
video_rec_mv = recall_score(video_labels, video_preds_mv)
video_f1_mv = f1_score(video_labels, video_preds_mv)

print("\n===== VIDEO-LEVEL RESULTS (MAJORITY VOTING) =====")
print(f"Accuracy:  {video_acc_mv:.4f}")
print(f"Precision: {video_prec_mv:.4f}")
print(f"Recall:    {video_rec_mv:.4f}")
print(f"F1 Score:  {video_f1_mv:.4f}")


===== VIDEO-LEVEL RESULTS (MAJORITY VOTING) =====
Accuracy:  0.9745
Precision: 0.9688
Recall:    0.8611
F1 Score:  0.9118


In [ ]:
# save video level results (from average probabilities)
video_results = pd.DataFrame({
    "video_id": list(video_dict.keys()),
    "label": video_labels,
    "avg_prob": [np.mean(video_dict[v]) for v in video_dict.keys()],
    "pred": video_preds
})

video_results.to_csv("video_level_results.csv", index=False)